# Problem 1: West Bengal State Election 2021 - Candidate Analysis Infographic
## Single-page infographic for prime-time TV news broadcast using Plotly

**Objective:** Create a comprehensive visual analysis of WB 2021 election candidates showcasing party distribution, criminal cases, education, and assets

**Author:** [Your Name]

**Date:** May 16, 2026

**Theme:** White background with #334455 color scheme

**Design:** TV-ready, clear at first glance, professional presentation

**Technology:** Plotly (NOT Dash - static infographic)

## Cell 1: Import Required Libraries

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Cell 2: Define Theme Colors

In [2]:
# Primary theme color (#345 expanded to #334455) and variations
THEME_PRIMARY = '#334455'      # Main theme color
THEME_DARK = '#223344'         # Darker shade for emphasis
THEME_LIGHT_1 = '#556677'      # Light shade 1
THEME_LIGHT_2 = '#778899'      # Light shade 2
THEME_LIGHT_3 = '#99AABB'      # Light shade 3
THEME_LIGHT_4 = '#BBCCDD'      # Light shade 4 (#cde equivalent)
THEME_LIGHT_5 = '#CCDDEE'      # Lightest shade

# Pastel colors for differentiation (light, soft colors)
COLOR_CLEAN_PASTEL = '#44DF44'      # Light pastel green for clean record
COLOR_CRIMINAL_PASTEL = '#FF3333'   # Light pastel red for criminal cases
COLOR_HIGHLIGHT = '#FFD3B6'         # Light pastel orange for highlights

# Background
BG_COLOR = '#FFFFFF'
GRID_COLOR = '#E8E8E8'

print("✓ Theme colors configured (#345 palette with gradients)")

✓ Theme colors configured (#345 palette with gradients)


## Cell 3: Load and Explore Data

In [3]:
# Load dataset
df = pd.read_csv('DSM301_MSE_PROB1_DATA.csv')

print(f"Dataset loaded: {len(df)} candidates")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

Dataset loaded: 567 candidates

Columns: ['candidate', 'constituency', 'party', 'criminal_cases', 'education', 'total_assets', 'liabilities']

First 5 rows:


,candidate,constituency,party,criminal_cases,education,total_assets,liabilities
0,Abdul Hai Mallik,ONDA,IND,0,8th Pass,160171,0
1,Abdur Razzak Molla,FALTA,INC,2,10th Pass,2650450,0
2,Abhijit Bhattacharya,PURULIA,IND,0,Post Graduate,4439532,375000
3,Abir Chandra Mandal,CHHATNA,IND,0,Graduate Professional,809000,0
4,Adhikari Suvendu,NANDIGRAM,BJP,1,Post Graduate,10552749,0


## Cell 4: Data Preprocessing

In [4]:
# Convert assets to Crores
df['total_assets_cr'] = df['total_assets'] / 10000000
df['liabilities_cr'] = df['liabilities'] / 10000000
df['net_worth_cr'] = df['total_assets_cr'] - df['liabilities_cr']
df['has_criminal_case'] = df['criminal_cases'] > 0

# Simplify education categories
def simplify_education(edu):
    if pd.isna(edu):
        return 'Unknown'
    edu = str(edu).strip()
    if 'Illiterate' in edu or '5th' in edu or '8th' in edu:
        return 'Below 10th'
    elif '10th' in edu or '12th' in edu:
        return '10th-12th'
    elif 'Graduate Professional' in edu:
        return 'Professional'
    elif 'Post Graduate' in edu or 'Doctorate' in edu:
        return 'Post Grad+'
    elif 'Graduate' in edu:
        return 'Graduate'
    else:
        return 'Other'

df['education_simplified'] = df['education'].apply(simplify_education)

# Group parties (top 6 + Others)
top_parties = df['party'].value_counts().head(6).index.tolist()
df['party_grouped'] = df['party'].apply(lambda x: x if x in top_parties else 'Others')

print("✓ Data preprocessed")
print(f"\nTop parties: {', '.join(top_parties)}")

✓ Data preprocessed

Top parties: IND, BJP, AITC, SUCI(C), CPI(M), BSP


## Cell 5: Calculate Key Statistics

In [5]:
# Overall statistics
total_candidates = len(df)
candidates_with_cases = df['has_criminal_case'].sum()
candidates_clean = total_candidates - candidates_with_cases
avg_assets = df['total_assets_cr'].mean()
median_assets = df['total_assets_cr'].median()

# Party statistics
party_counts = df['party_grouped'].value_counts()

# Education statistics
edu_order = ['Below 10th', '10th-12th', 'Graduate', 'Professional', 'Post Grad+', 'Other']
edu_stats = df['education_simplified'].value_counts()
edu_counts = [edu_stats.get(edu, 0) for edu in edu_order]

# Asset distribution
asset_bins = [0, 0.5, 1, 2, 5, 10, df['total_assets_cr'].max()]
asset_labels = ['<50L', '50L-1Cr', '1-2Cr', '2-5Cr', '5-10Cr', '>10Cr']
df['asset_range'] = pd.cut(df['total_assets_cr'], bins=asset_bins, labels=asset_labels)
asset_counts = df['asset_range'].value_counts().sort_index()

# Criminal cases by party
top5_parties = df['party_grouped'].value_counts().head(5).index
party_criminal_data = []
for party in top5_parties:
    party_df = df[df['party_grouped'] == party]
    total = len(party_df)
    with_cases = party_df['has_criminal_case'].sum()
    party_criminal_data.append({
        'party': party,
        'clean': total - with_cases,
        'criminal': with_cases,
        'total': total,
        'criminal_pct': (with_cases / total * 100) if total > 0 else 0
    })
party_criminal_df = pd.DataFrame(party_criminal_data)

print("✓ Statistics calculated")
print(f"\nTotal: {total_candidates} | With Cases: {candidates_with_cases} | Clean: {candidates_clean}")
print(f"Avg Assets: ₹{avg_assets:.2f} Cr | Median: ₹{median_assets:.2f} Cr")

✓ Statistics calculated

Total: 567 | With Cases: 144 | Clean: 423
Avg Assets: ₹0.71 Cr | Median: ₹0.17 Cr


## Cell 6: Create Plotly Infographic with Subplots

In [6]:
# Create subplot layout: 3 rows x 3 columns
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=(
        'Party-wise Candidate Distribution', 'Criminal Cases Status', 'Education Qualification',
         'Asset Distribution', 'Criminal Cases by Party',
        '', '', '', ''
    ),
    specs=[
        [{"rowspan": 2, "type": "bar"}, {"type": "pie"}, {"type": "bar"}],
        [None, {"type": "bar"}, {"type": "bar"}],
        [{"colspan": 3, "type": "table"}, None, None]
    ],
    vertical_spacing=0.15,
    horizontal_spacing=0.12,
    row_heights=[0.38, 0.32, 0.30]
)

print("✓ Subplot structure created")

✓ Subplot structure created


## Cell 7: Add Visualizations to Subplots

In [7]:
# VIZ 1: Party Distribution (Row 1-2, Col 1)
# Gradient from dark theme (top/highest) to light theme (bottom/lowest)
colors_party = [THEME_PRIMARY, THEME_PRIMARY, THEME_LIGHT_1, 
                THEME_LIGHT_2, THEME_LIGHT_3, THEME_LIGHT_4, THEME_LIGHT_5]

fig.add_trace(
    go.Bar(
        y=party_counts.index,
        x=party_counts.values,
        orientation='h',
        marker=dict(color=colors_party[:len(party_counts)], line=dict(color=THEME_DARK, width=2)),
        text=party_counts.values,
        textposition='outside',
        textfont=dict(size=12, color=THEME_DARK, family='Arial Black'),
        hovertemplate='<b>%{y}</b><br>Candidates: %{x}<extra></extra>',
        showlegend=False
    ),
    row=1, col=1
)

# VIZ 2: Criminal Cases Pie Chart (Row 1, Col 2)
# Using light pastel colors with clear labels
fig.add_trace(
    go.Pie(
        labels=['Clean Record', 'Criminal Cases'],
        values=[candidates_clean, candidates_with_cases],
        marker=dict(
            colors=[COLOR_CLEAN_PASTEL, COLOR_CRIMINAL_PASTEL],
            line=dict(color=THEME_DARK, width=2.5)
        ),
        text=[f'{candidates_clean}', f'{candidates_with_cases}'],
        textinfo='label+text+percent',
        textposition='inside',
        textfont=dict(size=12, color=THEME_DARK, family='Arial Black'),
        hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>',
        showlegend=False
    ),
    row=1, col=2
)

# VIZ 3: Education Levels (Row 1, Col 3)
# Gradient from dark (highest) to light (lowest)
colors_edu = [THEME_PRIMARY, THEME_PRIMARY, THEME_LIGHT_1, 
              THEME_LIGHT_2, THEME_LIGHT_3, THEME_LIGHT_4]

fig.add_trace(
    go.Bar(
        x=edu_order,
        y=edu_counts,
        marker=dict(color=colors_edu, line=dict(color=THEME_DARK, width=2)),
        text=edu_counts,
        textposition='outside',
        textfont=dict(size=11, color=THEME_DARK, family='Arial Black'),
        hovertemplate='<b>%{x}</b><br>Candidates: %{y}<extra></extra>',
        showlegend=False
    ),
    row=1, col=3
)

# VIZ 4: Asset Distribution (Row 2, Col 2)
# Gradient from light (low assets) to dark (high assets)
colors_assets = [THEME_LIGHT_4, THEME_LIGHT_3, THEME_LIGHT_2, 
                 THEME_LIGHT_1, THEME_PRIMARY, THEME_DARK]

fig.add_trace(
    go.Bar(
        y=asset_counts.index,
        x=asset_counts.values,
        orientation='h',
        marker=dict(color=colors_assets, line=dict(color=THEME_DARK, width=2)),
        text=asset_counts.values,
        textposition='outside',
        textfont=dict(size=11, color=THEME_DARK, family='Arial Black'),
        hovertemplate='<b>%{y}</b><br>Candidates: %{x}<extra></extra>',
        showlegend=False
    ),
    row=2, col=2
)

# VIZ 5: Criminal Cases by Party (Row 2, Col 3)
fig.add_trace(
    go.Bar(
        x=party_criminal_df['party'],
        y=party_criminal_df['clean'],
        name='Clean Record',
        marker=dict(color=COLOR_CLEAN_PASTEL, line=dict(color=THEME_DARK, width=2)),
        hovertemplate='<b>%{x}</b><br>Clean: %{y}<extra></extra>',
        showlegend=True
    ),
    row=2, col=3
)

fig.add_trace(
    go.Bar(
        x=party_criminal_df['party'],
        y=party_criminal_df['criminal'],
        name='Criminal Cases',
        marker=dict(color=COLOR_CRIMINAL_PASTEL, line=dict(color=THEME_DARK, width=2)),
        hovertemplate='<b>%{x}</b><br>Criminal Cases: %{y}<extra></extra>',
        showlegend=True
    ),
    row=2, col=3
)

# Add percentage annotations with better positioning
for i, row in party_criminal_df.iterrows():
    if row['criminal'] > 0:
        fig.add_annotation(
            x=i, y=row['total'] + 2.5,
            text=f"{row['criminal_pct']:.0f}%",
            showarrow=False,
            font=dict(size=11, color=COLOR_CRIMINAL_PASTEL, family='Arial Black'),
            row=2, col=3
        )

print("✓ All visualizations added")

✓ All visualizations added


## Cell 8: Add Statistics Table

In [8]:
# Calculate insights
criminal_pct = (candidates_with_cases / total_candidates) * 100
highest_criminal_party = party_criminal_df.nlargest(1, 'criminal_pct')['party'].values[0]
highest_criminal_pct = party_criminal_df.nlargest(1, 'criminal_pct')['criminal_pct'].values[0]
most_common_edu = df['education_simplified'].value_counts().index[0]
most_common_edu_count = df['education_simplified'].value_counts().values[0]
high_asset_candidates = len(df[df['total_assets_cr'] > 5])
top_party = party_counts.index[0]
top_party_count = party_counts.values[0]

# Create statistics table
stats_data = [
    ['<b>KEY STATISTICS</b>', '<b>VALUE</b>', '<b>INSIGHT</b>'],
    ['Total Candidates', f'{total_candidates}', f'{len(top_parties)} major parties competing'],
    ['Criminal Cases', f'{candidates_with_cases} ({criminal_pct:.1f}%)', f'{highest_criminal_party} has highest at {highest_criminal_pct:.0f}%'],
    ['Clean Record', f'{candidates_clean} ({100-criminal_pct:.1f}%)', 'Majority have clean background'],
    ['Average Assets', f'₹{avg_assets:.2f} Cr', f'{high_asset_candidates} candidates have >₹5 Cr'],
    ['Median Assets', f'₹{median_assets:.2f} Cr', 'Significant wealth disparity exists'],
    ['Top Party', f'{top_party}', f'Fields {top_party_count} candidates'],
    ['Most Common Education', f'{most_common_edu}', f'{most_common_edu_count} candidates']
]

fig.add_trace(
    go.Table(
        header=dict(
            values=stats_data[0],
            fill_color=THEME_PRIMARY,
            align='left',
            font=dict(color='white', size=13, family='Arial Black'),
            height=35
        ),
        cells=dict(
            values=list(zip(*stats_data[1:])),
            fill_color=[['#F5F5F5', 'white'] * 4],
            align='left',
            font=dict(color=THEME_DARK, size=12, family='Arial'),
            height=30
        )
    ),
    row=3, col=1
)

print("✓ Statistics table added")

✓ Statistics table added


## Cell 9: Update Layout and Styling

In [9]:
# Update axes
fig.update_xaxes(showgrid=True, gridcolor=GRID_COLOR, gridwidth=1, zeroline=False,
                 showline=True, linewidth=2, linecolor=THEME_LIGHT_1)
fig.update_yaxes(showgrid=True, gridcolor=GRID_COLOR, gridwidth=1, zeroline=False,
                 showline=True, linewidth=2, linecolor=THEME_LIGHT_1)

# Update specific axes labels
fig.update_xaxes(title_text="Number of Candidates", 
                 title_font=dict(size=12, color=THEME_DARK, family='Arial Black'), row=1, col=1)
fig.update_xaxes(title_text="Education Level", 
                 title_font=dict(size=12, color=THEME_DARK, family='Arial Black'), row=1, col=3)
fig.update_yaxes(title_text="Number of Candidates", 
                 title_font=dict(size=12, color=THEME_DARK, family='Arial Black'), row=1, col=3)
fig.update_xaxes(title_text="Number of Candidates", 
                 title_font=dict(size=12, color=THEME_DARK, family='Arial Black'), row=2, col=2)
fig.update_xaxes(title_text="Party", 
                 title_font=dict(size=12, color=THEME_DARK, family='Arial Black'), row=2, col=3)
fig.update_yaxes(title_text="Number of Candidates", 
                 title_font=dict(size=12, color=THEME_DARK, family='Arial Black'), row=2, col=3)

# Update layout with proper spacing for title and salient points
fig.update_layout(
    title={
        'text': '<b>WEST BENGAL STATE ELECTION 2021 | Candidate Profile Analysis</b>',
        'x': 0.5,
        'xanchor': 'center',
        'y': 0.98,
        'yanchor': 'top',
        'font': {'size': 24, 'color': THEME_DARK, 'family': 'Arial Black'}
    },
    showlegend=True,
    legend=dict(
        orientation="h", yanchor="top", y=-0.02, xanchor="center", x=0.75,
        font=dict(size=11, family='Arial Black'),
        bgcolor='rgba(255,255,255,0.8)', bordercolor=THEME_DARK, borderwidth=2
    ),
    height=1700, width=1900,
    plot_bgcolor=BG_COLOR, paper_bgcolor=BG_COLOR,
    font=dict(family='Arial', size=11, color=THEME_DARK),
    barmode='stack',
    margin=dict(t=100, b=500, l=100, r=100)
)

# Update subplot title styling
for annotation in fig['layout']['annotations'][:6]:
    annotation['font'] = dict(size=14, color=THEME_DARK, family='Arial Black')

print("✓ Layout and styling updated")

✓ Layout and styling updated


## Cell 10: Add Salient Points Annotation

In [10]:
salient_points_text = f"""<b>3 SALIENT POINTS FOR PRIME-TIME REPORTER:</b><br><br>
<b>1. CRIMINAL BACKGROUND CONCERN:</b> Nearly {criminal_pct:.0f}% of candidates ({candidates_with_cases} out of {total_candidates})<br>
have criminal cases pending against them. {highest_criminal_party} has the highest proportion at {highest_criminal_pct:.0f}%<br>
of their candidates.<br><br>
<b>2. EDUCATION PROFILE:</b> {most_common_edu} is the most common education level with {most_common_edu_count} candidates.<br>
However, a significant number of candidates have education below 10th standard, raising questions about<br>
candidate qualifications.<br><br>
<b>3. WEALTH CONCENTRATION:</b> The median candidate asset is ₹{median_assets:.2f} Crores, but {high_asset_candidates} candidates<br>
have assets exceeding ₹5 Crores. {top_party} fields the most candidates ({top_party_count}), dominating the<br>
electoral landscape."""

fig.add_annotation(
    text=salient_points_text,
    xref="paper", yref="paper",
    x=0.5, y=-0.18,
    xanchor='center', yanchor='top',
    showarrow=False,
    font=dict(size=11, color=THEME_DARK, family='Arial'),
    align='left',
    bgcolor='#F5F5F5',
    bordercolor=THEME_PRIMARY,
    borderwidth=3,
    borderpad=25
)

print("✓ Salient points added with proper spacing")

✓ Salient points added with proper spacing


## Cell 11: Display the Infographic

In [11]:
# Display the infographic
fig.show()

print("\n" + "="*80)
print("PLOTLY INFOGRAPHIC COMPLETE")
print("="*80)
print("\nTheme: #334455 color palette with gradient shades")
print("Colors: Light pastel green/red for criminal cases differentiation")
print("Technology: Plotly (NOT Dash)")
print("3 Salient Points included for prime-time reporter")
print("="*80)


PLOTLY INFOGRAPHIC COMPLETE

Theme: #334455 color palette with gradient shades
Colors: Light pastel green/red for criminal cases differentiation
Technology: Plotly (NOT Dash)
3 Salient Points included for prime-time reporter


## Cell 12: Save the Infographic (Optional)

In [12]:
# Save as HTML (interactive)
html_file = 'WB_Election_2021_Infographic.html'
fig.write_html(html_file)
print(f"✓ Interactive HTML saved as: {html_file}")

# Uncomment below to save as PNG (requires kaleido package)
# png_file = 'WB_Election_2021_Infographic.png'
# fig.write_image(png_file, width=1900, height=1700, scale=2)
# print(f"✓ Static PNG saved as: {png_file}")

✓ Interactive HTML saved as: WB_Election_2021_Infographic.html


---

## Summary

This notebook creates a comprehensive single-page infographic for WB Election 2021 using **Plotly** (not Dash). The visualization includes:

### Visualizations:
1. **Party-wise Distribution** - Horizontal bar chart with gradient (dark #345 for top parties, lighter shades for others)
2. **Criminal Cases Status** - Pie chart with light pastel colors (green for clean, red for criminal cases)
3. **Education Qualification** - Bar chart with gradient shading
4. **Asset Distribution** - Horizontal bar chart with proper text spacing
5. **Criminal Cases by Party** - Stacked bar chart with pastel colors
6. **Key Statistics Table** - Comprehensive data summary
7. **3 Salient Points** - Critical insights with proper border spacing

### Design Improvements:
- ✅ Light pastel colors (green/red) for criminal cases with clear labels
- ✅ Gradient shading using #345 theme (dark for highest, light for lowest)
- ✅ Fixed overlapping numbers in asset distribution
- ✅ Proper spacing for title (no overlap with subplots)
- ✅ Proper spacing in salient points annotation (all text visible)
- ✅ White background with professional styling
- ✅ TV-ready, clear at first glance
- ✅ High contrast for broadcast quality